# Projet NOZ - Étape 5 : prédire les RVN prématurés

David DOE · 2026

La question métier : quand un lot arrive, peut-on deviner s'il risque d'être renvoyé avant la fin des 7 semaines (RVN prématuré) ?
Si oui, le manager peut agir tôt : mieux le placer, le remiser plus vite, ou refuser de le saturer.

Le plan :
1. Préparer les données sans tricher (pas de fuite de données).
2. Couper le temps en deux : on apprend sur le passé, on teste sur le futur.
3. Comparer 3 approches : une règle simple (référence), une régression logistique, une forêt aléatoire (Random Forest).
4. Deux moments de prédiction : le jour de l'arrivée, puis après 7 jours de vente.
5. Lire les résultats en langage métier.

 Données simulées : on valide la méthode, on ne décrit pas le vrai magasin.

Installation (une seule fois) dans le terminal de VS Code :
```
python -m pip install scikit-learn
```

## 1. Connexion et vocabulaire

- Cible (ce qu'on veut prédire) : 1 si le lot finit en RVN prématuré, 0 sinon.
- Variables (les indices qu'on donne au modèle) : catégorie, prix, saturation du magasin...
- Modèle : une « machine » qui apprend le lien entre les variables et la cible sur des exemples passés.
- scikit-learn (`sklearn`) : la bibliothèque Python de référence pour le machine learning.

In [ ]:
import os, getpass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, URL
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

url = os.environ.get("NOZ_DB_URL")
if url is None:
    url = URL.create("postgresql+psycopg2", username="postgres",
                     password=getpass.getpass("Mot de passe PostgreSQL : "),
                     host="localhost", port=5432, database="noz")
moteur = create_engine(url)
os.makedirs("figures", exist_ok=True)
print("Connexion OK")

## 2. Préparer les données SANS fuite

La règle d'or : on ne donne au modèle que ce qu'on savait au moment de la décision.
Le jour de l'arrivée, on connaît le lot (catégorie, prix, quantité...) et l'état du magasin ce jour-là. On ne connaît pas encore ses ventes, ni comment il finira.

Exemple de triche : donner au modèle « le nombre de jours passés en rayon ». Un lot renvoyé tôt a forcément peu de jours : le modèle aurait 100 % de réussite… et serait inutile dans la vraie vie.

Quels lots garder ?
- Seulement ceux arrivés en 2024 ou après (on connaît leur début de vie).
- Seulement ceux arrivés au moins 49 jours avant la fin des données (21/08/2025), soit jusqu'au 04/07/2025.

Pourquoi la 2e règle ? C'est le piège de la censure. Un lot arrivé fin juillet 2025 et déjà sorti au 21/08 est forcément sorti vite, donc presque toujours parce qu'il a été écoulé. Si on le garde, on fausse la cible. Avec la règle des 49 jours, chaque lot a eu toute sa vie pour finir.

In [ ]:
requete = """
SELECT a.arrivage_id,
       a.date_arrivee,
       -- Ce qu'on sait LE JOUR DE L'ARRIVEE
       a.categorie, a.type_marque, a.indice_fragilite,
       a.nb_palettes, a.quantite_initiale,
       a.prix_unitaire_initial::float   AS prix,
       a.taux_saturation_arrivee::float AS saturation_arrivee,
       m.jour_semaine                   AS jour_arrivee,
       m.evenement, m.effectif, m.nb_lots_actifs,
       m.stock_theorique_unites         AS stock_magasin,
       EXTRACT(MONTH FROM a.date_arrivee)::int AS mois,
       -- Ce qu'on sait APRES 7 JOURS (pour le 2e modele)
       s1.jours_vus_s1, s1.vendu_s1, s1.saturation_moy_s1,
       -- La CIBLE (connue seulement a la fin : jamais une variable !)
       (s.motif_sortie = 'RVN_PREMATURE')::int AS cible
FROM arrivages a
JOIN sorties_lots s ON s.arrivage_id = a.arrivage_id
JOIN magasin_jour m ON m.date_jour   = a.date_arrivee
LEFT JOIN (SELECT v.arrivage_id,
                  COUNT(*)                      AS jours_vus_s1,
                  SUM(v.quantite_vendue)        AS vendu_s1,
                  AVG(mj.taux_saturation)::float AS saturation_moy_s1
           FROM ventes_lot_jour v
           JOIN magasin_jour mj ON mj.date_jour = v.date_jour
           WHERE v.jour_de_vie <= 7
           GROUP BY v.arrivage_id) s1 ON s1.arrivage_id = a.arrivage_id
WHERE a.arrive_dans_fenetre
  AND a.date_arrivee <= DATE '2025-08-21' - 48     -- regle anti-censure
"""
lots = pd.read_sql(requete, moteur)
lots["date_arrivee"] = pd.to_datetime(lots["date_arrivee"])
print(f"{len(lots)} lots, dont {lots['cible'].sum()} RVN prematures ({lots['cible'].mean():.1%})")
lots.head()

## 3. Couper le temps : apprendre sur le passé, tester sur le futur

On n'utilise pas un découpage au hasard. Le modèle servira à prédire l'avenir, donc on le teste comme ça :
- Entraînement : lots arrivés avant le 01/02/2025 (le modèle apprend).
- Test : lots arrivés à partir du 01/02/2025 (le modèle n'a jamais vu ces lots ; on note sa copie).

C'est comme réviser avec les annales des années passées, puis passer l'examen de cette année.

In [ ]:
DATE_COUPURE = pd.Timestamp("2025-02-01")
lots["periode"] = np.where(lots["date_arrivee"] < DATE_COUPURE, "entrainement", "test")
lots.groupby("periode")["cible"].agg(nb_lots="size", taux_rvn="mean")

À remarquer : le taux de RVN prématuré est plus élevé dans la période de test que pendant l'entraînement : le magasin a été plus saturé début 2025.
Le modèle doit donc prédire dans un contexte un peu différent de celui qu'il a appris. C'est réaliste, et c'est exactement ce qu'un découpage au hasard aurait caché.

## 4. Comment on note un modèle : l'AUC

Le modèle donne à chaque lot un score de risque entre 0 et 1.

L'AUC répond à : *si je prends au hasard un lot renvoyé et un lot non renvoyé, quelle est la chance que le modèle donne un score plus élevé au lot renvoyé ?*
- AUC = 0,5 : pas mieux qu'une pièce à pile ou face.
- AUC = 1 : le modèle classe parfaitement.
- Objectif du cahier des charges : AUC ≥ 0,75.

Pourquoi pas le « pourcentage de bonnes réponses » ? Seulement ~11 % des lots sont des RVN. Un modèle qui répond toujours « pas de RVN » aurait 89 % de bonnes réponses… sans servir à rien.

## 5. Modèle A : prédire le jour de l'arrivée

On compare trois approches :
1. Règle simple (référence) : « plus le magasin est saturé à l'arrivée, plus le risque est grand ». C'est la règle de l'ancienne v1.
2. Régression logistique : elle donne un « poids » à chaque variable et les additionne. Simple et facile à expliquer.
3. Random Forest : 500 arbres de décision (des suites de questions oui/non) qui votent. Plus puissant, mais moins lisible.

Deux préparations nécessaires :
- `OneHotEncoder` : un modèle ne comprend que des nombres. « Catégorie = Boissons » devient une colonne 0/1 « est_Boissons ».
- `StandardScaler` : met les nombres à la même échelle (un prix de 5 € et un stock de 60 000 unités), sinon la régression logistique est déséquilibrée.

In [ ]:
VAR_NUM_ARRIVEE = ["nb_palettes", "quantite_initiale", "prix", "saturation_arrivee",
                   "effectif", "nb_lots_actifs", "stock_magasin", "mois"]
VAR_CAT = ["categorie", "type_marque", "indice_fragilite", "jour_arrivee", "evenement"]

def entrainer_et_noter(donnees, var_num, var_cat, nom):
    """Entraine les 2 modeles sur le passe, les note sur le futur."""
    train = donnees[donnees["periode"] == "entrainement"]
    test  = donnees[donnees["periode"] == "test"]
    X = var_num + var_cat

    logit = make_pipeline(
        make_column_transformer((StandardScaler(), var_num),
                                (OneHotEncoder(handle_unknown="ignore"), var_cat)),
        LogisticRegression(max_iter=2000))
    foret = make_pipeline(
        make_column_transformer(("passthrough", var_num),
                                (OneHotEncoder(handle_unknown="ignore"), var_cat)),
        RandomForestClassifier(n_estimators=500, min_samples_leaf=10, random_state=42, n_jobs=-1))
    logit.fit(train[X], train["cible"])
    foret.fit(train[X], train["cible"])

    scores = {
        "Regle simple (saturation)": test["saturation_arrivee"],
        "Regression logistique":     logit.predict_proba(test[X])[:, 1],
        "Random Forest":             foret.predict_proba(test[X])[:, 1],
    }
    resultats = pd.DataFrame({
        "modele": list(scores),
        "AUC_test": [round(roc_auc_score(test["cible"], s), 3) for s in scores.values()],
    })
    # Verifier le sur-apprentissage : la note sur les donnees d'entrainement
    resultats["AUC_entrainement"] = [
        round(roc_auc_score(train["cible"], train["saturation_arrivee"]), 3),
        round(roc_auc_score(train["cible"], logit.predict_proba(train[X])[:, 1]), 3),
        round(roc_auc_score(train["cible"], foret.predict_proba(train[X])[:, 1]), 3)]
    print(f"--- {nom} : {len(train)} lots pour apprendre, {len(test)} lots pour tester ---")
    return resultats, scores, logit, test, X

res_A, scores_A, logit_A, test_A, X_A = entrainer_et_noter(lots, VAR_NUM_ARRIVEE, VAR_CAT, "Modele A (arrivee)")
res_A

Comment lire ce tableau :
- La règle simple fait à peine mieux que le hasard : la saturation seule ne suffit pas (on l'avait vu avec H4).
- Les deux modèles dépassent l'objectif de 0,75 : combiner plusieurs indices fonctionne.
- AUC_entrainement = la note sur les annales. Si elle est beaucoup plus haute que la note de test, le modèle a « appris par cœur » au lieu de comprendre : c'est le sur-apprentissage. La Random Forest y est plus sujette.
- Le modèle le plus simple (régression logistique) fait aussi bien voire mieux que la Random Forest. Le plus compliqué n'est pas toujours le meilleur. En entreprise, on choisirait le plus simple, car il est plus facile à expliquer.

## 6. Modèle B : prédire après 7 jours de vente

Ton mémoire recommandait de surveiller les lots dès les premières semaines. Après 7 jours, on sait en plus :
- le taux de vente de la 1re semaine (KPI du magasin) ;
- la saturation moyenne du magasin pendant cette semaine.

Ce n'est pas de la triche : au 7e jour, ces informations sont réellement connues, et un RVN prématuré n'arrive jamais avant le 14e jour.

On garde seulement les lots encore en rayon au 7e jour (ceux déjà écoulés n'ont plus besoin de prédiction).

In [ ]:
lots_j7 = lots[lots["jours_vus_s1"] == 7].copy()
lots_j7["taux_vente_s1"] = lots_j7["vendu_s1"] / lots_j7["quantite_initiale"]
print(f"{len(lots_j7)} lots encore en rayon au jour 7 ({len(lots) - len(lots_j7)} deja ecoules)")

VAR_NUM_J7 = VAR_NUM_ARRIVEE + ["taux_vente_s1", "saturation_moy_s1"]
res_B, scores_B, logit_B, test_B, X_B = entrainer_et_noter(lots_j7, VAR_NUM_J7, VAR_CAT, "Modele B (jour 7)")
res_B

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (titre, scores, test) in zip(axes, [("A - le jour de l'arrivee", scores_A, test_A),
                                            ("B - apres 7 jours", scores_B, test_B)]):
    for nom, s in scores.items():
        fpr, tpr, _ = roc_curve(test["cible"], s)
        ax.plot(fpr, tpr, label=f"{nom} (AUC {roc_auc_score(test['cible'], s):.2f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="hasard (AUC 0,50)")
    ax.set_title(f"Modele {titre}")
    ax.set_xlabel("% de lots sains signales a tort")
    ax.set_ylabel("% de vrais RVN detectes")
    ax.legend(fontsize=8, loc="lower right")
plt.suptitle("Courbes ROC sur la periode de test (donnees simulees)")
plt.tight_layout()
plt.savefig("figures/m5_courbes_roc.png", dpi=150)
plt.show()

Lire une courbe ROC : plus la courbe monte vite vers le coin en haut à gauche, meilleur est le modèle. La diagonale en pointillés, c'est le hasard. L'AUC est la surface sous la courbe.

Conclusion : attendre 7 jours améliore nettement la prédiction. Le taux de vente de la 1re semaine est un signal très fort, ce qui confirme l'intuition de ton mémoire.

## 7. Traduire en décision pour le manager

Supposons que l'équipe peut surveiller 1 lot sur 5 (les 20 % aux scores les plus élevés). Combien de vrais RVN prématurés seront dans cette liste ?

In [ ]:
def liste_a_surveiller(test, scores, part=0.20):
    seuil = np.quantile(scores, 1 - part)
    signale = scores >= seuil
    vrais_rvn = test["cible"].to_numpy() == 1
    detectes = (signale & vrais_rvn).sum()
    return {"lots signales": int(signale.sum()),
            "vrais RVN dans la liste": int(detectes),
            "RVN total": int(vrais_rvn.sum()),
            "% des RVN detectes (rappel)": f"{detectes / vrais_rvn.sum():.0%}",
            "% de la liste qui sont des RVN (precision)": f"{detectes / signale.sum():.0%}"}

pd.DataFrame({
    "Hasard (1 lot sur 5)": {"% des RVN detectes (rappel)": "20%"},
    "Modele A - arrivee": liste_a_surveiller(test_A, scores_A["Regression logistique"]),
    "Modele B - jour 7":  liste_a_surveiller(test_B, scores_B["Regression logistique"]),
}).fillna("")

Deux mots à connaître :
- Rappel : parmi tous les vrais RVN, combien on en attrape ? (« on ne rate rien »)
- Précision : parmi les lots qu'on signale, combien sont vraiment des RVN ? (« on ne fait pas perdre de temps »)

En surveillant seulement 20 % des lots, le modèle B attrape environ 2 futurs RVN sur 3 (65 %), et 2 lots signalés sur 3 sont de vrais RVN. Au hasard, on n'en attraperait que 20 %, soit 3 fois moins.

## 8. Qu'est-ce qui fait monter le risque ?

Importance par permutation : on « mélange » une variable au hasard et on regarde de combien l'AUC baisse. Si elle baisse beaucoup, la variable était importante. Si rien ne change, le modèle ne s'en servait pas.

In [ ]:
imp = permutation_importance(logit_B, test_B[X_B], test_B["cible"],
                             scoring="roc_auc", n_repeats=20, random_state=42)
importance = (pd.DataFrame({"variable": X_B, "baisse_AUC": imp.importances_mean})
              .sort_values("baisse_AUC", ascending=False))
importance["baisse_AUC"] = importance["baisse_AUC"].round(3)

top = importance.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top["variable"], top["baisse_AUC"], color="#2c3e50")
ax.set_xlabel("Baisse de l'AUC quand on melange la variable")
ax.set_title("Modele B - variables les plus utiles (donnees simulees)")
plt.tight_layout()
plt.savefig("figures/m5_importance_variables.png", dpi=150)
plt.show()
importance.head(8)

Attention à la lecture : une variable importante aide à prédire, mais ça ne prouve pas qu'elle cause le RVN. Par exemple, un faible taux de vente en semaine 1 n'est pas « la cause » : c'est le symptôme d'un lot qui plaît peu.

## 9. Synthèse (pour le rapport, section 5.2)

In [ ]:
synthese = pd.concat([res_A.assign(moment="A - arrivee"), res_B.assign(moment="B - jour 7")])
synthese[["moment", "modele", "AUC_test", "AUC_entrainement"]]

> « Avec un découpage temporel et des variables disponibles au moment de la décision, une régression logistique prédit les RVN prématurés avec une AUC d'environ 0,84 dès l'arrivée du lot, et d'environ 0,92 après une semaine de vente. Elle fait mieux qu'une Random Forest et bien mieux que la règle de saturation seule (≈ 0,5). »

Limites:
- Données simulées : le modèle retrouve les règles du simulateur.
- Le taux de RVN change dans le temps (dérive) : il faudrait réentraîner le modèle régulièrement.
- Un RVN décidé par le manager dépend aussi de facteurs qu'on n'a pas (place disponible ce jour-là, consignes du siège).

